<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.2/blob/main/%E3%80%87Language_geriven_exproration_Viollin_plot_2605.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================
# Complete code:
# BO-consistent Violin + JSD diversity plot
# with A4 Word export
#
# Required:
#   numeric_matrix.csv
#   WordToken_features_concept_only.csv
# ===============================================================

!pip -q install scikit-learn scipy matplotlib pandas numpy openpyxl python-docx

import os, glob, re, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.ensemble import GradientBoostingRegressor
from scipy.spatial.distance import cdist

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.section import WD_SECTION
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

warnings.filterwarnings("ignore")
np.random.seed(0)

# ===============================================================
# 0. Settings
# ===============================================================

OUTDIR = "./BO_consistent_violin_JSD"
os.makedirs(OUTDIR, exist_ok=True)

ZIP_PATH = "./BO_consistent_violin_JSD.zip"
WORD_PATH = "./BO_consistent_violin_JSD_A4.docx"

N_REPEATS = 5
BO_ITERS = 40
BLOX_ITERS = 40

N_POOL = 800
N_CAND = 600

TOTAL_ACTIVE_DEFAULT = 0.97
DYNAMICS_SOURCE_VARIANT = "snd"

NUMERIC_WEIGHT = 1.0
LANGUANGE_WEIGHT = 1.0
MIN_COMPOSITION_DISTANCE = 0.03

TARGET_VARIANTS = ["n", "sn", "snc", "snm", "sni", "snd", "snp", "sncmidp"]

VARIANT_LABEL_MAP = {
    "n": "N",
    "sn": "N+S",
    "snc": "N+SC",
    "snm": "N+SM",
    "sni": "N+SI",
    "snd": "N+SD",
    "snp": "N+SP",
    "sncmidp": "N+SCMIDP",
}

METHODS = [
    "BO-Numerical",
    "BO-N+Languange",
    "BLOX-Numerical",
    "BLOX-N+Languange",
]

METHOD_COLORS = {
    "BO-Numerical": "#4D4D4D",
    "BO-N+Languange": "#0072B2",
    "BLOX-Numerical": "#009E73",
    "BLOX-N+Languange": "#E69F00",
}

HYDROPHILIC = ["AMPS", "pSSA", "MEDSAH", "VBA", "NIPAM", "HEA", "DMAA", "AAm"]
HYDROPHOBIC = ["HMA", "TFEMA", "HA"]
CROSSLINKER = ["TECL", "TRCL", "DICL"]
ALL_MONOMERS = sorted(set(HYDROPHILIC + HYDROPHOBIC + CROSSLINKER))

VIOLIN_MONOMERS = [
    "AMPS", "pSSA", "MEDSAH", "VBA", "NIPAM", "HEA",
    "HMA", "TFEMA", "HA",
    "TECL", "TRCL", "DICL"
]
VIOLIN_MONOMERS = [m for m in VIOLIN_MONOMERS if m in ALL_MONOMERS]

NUMERIC_REPR_COLS_BASE = [
    "comp_1", "comp_2", "comp_3",
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
    "motif_score_max",
    "motif_breadth_count_0p5",
    "motif_selectivity_index",
    "Peak1_T2_ms",
    "Peak2_T2_ms",
    "Peak3_T2_ms",
    "Peak4_T2_ms",
    "Width_log10T2",
    "Weighted_logmean_T2_ms",
    "N_detected_peaks",
    "Fit_R2",
    "Ridge_alpha",
]

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

LABEL_FS = 16
TICK_FS = 13
TITLE_FS = 16
SUBTITLE_FS = 13

GENERATED_FIGURES = []

# ===============================================================
# 1. File search
# ===============================================================

def find_file(patterns):
    hits = []
    for p in patterns:
        hits.extend(glob.glob(p, recursive=True))
    hits = sorted(set(hits), key=lambda x: os.path.getmtime(x), reverse=True)
    return hits[0] if hits else None

NUMERIC_CSV = find_file([
    "./**/numeric_matrix.csv",
    "/content/**/numeric_matrix.csv",
    "/mnt/data/**/numeric_matrix.csv",
])

LANGUANGE_CSV = find_file([
    "./**/WordToken_features_concept_only.csv",
    "./**/WordToken_features_concept_only*.csv",
    "/content/**/WordToken_features_concept_only.csv",
    "/content/**/WordToken_features_concept_only*.csv",
    "/mnt/data/**/WordToken_features_concept_only.csv",
    "/mnt/data/**/WordToken_features_concept_only*.csv",
])

if NUMERIC_CSV is None or LANGUANGE_CSV is None:
    from google.colab import files
    print("Please upload numeric_matrix.csv and WordToken_features_concept_only.csv")
    files.upload()

    NUMERIC_CSV = find_file(["./**/numeric_matrix.csv", "/content/**/numeric_matrix.csv"])
    LANGUANGE_CSV = find_file(["./**/WordToken_features_concept_only*.csv", "/content/**/WordToken_features_concept_only*.csv"])

if NUMERIC_CSV is None:
    raise FileNotFoundError("numeric_matrix.csv not found.")
if LANGUANGE_CSV is None:
    raise FileNotFoundError("WordToken_features_concept_only.csv not found.")

print("NUMERIC_CSV   :", NUMERIC_CSV)
print("LANGUANGE_CSV :", LANGUANGE_CSV)

# ===============================================================
# 2. Load data
# ===============================================================

numeric_df = pd.read_csv(NUMERIC_CSV, encoding="utf-8-sig")
languange_df = pd.read_csv(LANGUANGE_CSV, encoding="utf-8-sig")

META_COLS = ["row_id", "Copolymer_Name", "variant"]

for c in META_COLS:
    if c not in numeric_df.columns:
        raise KeyError(f"Missing column in numeric_matrix.csv: {c}")
    if c not in languange_df.columns:
        raise KeyError(f"Missing column in WordToken_features_concept_only.csv: {c}")

df = numeric_df.merge(languange_df, on=META_COLS, how="inner", suffixes=("", "_languangedup"))

LANGUANGE_FEATURE_COLS = [
    c for c in languange_df.columns
    if c.startswith("TOK_") or c.startswith("SP_")
]

if len(LANGUANGE_FEATURE_COLS) == 0:
    raise ValueError("No TOK_ or SP_ columns found.")

for c in NUMERIC_REPR_COLS_BASE:
    if c not in df.columns:
        df[c] = np.nan
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

# ===============================================================
# 3. Helper functions
# ===============================================================

def normalize_name(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"\(.*?\)", "", s)
    s = s.replace("-", "_").replace("/", "_")
    s = re.sub(r"\s+", "", s)
    s = s.replace("4VBA", "VBA").replace("MEDSH", "MEDSAH").replace("TECL2", "TECL")
    parts = s.split("_")
    if len(parts) >= 3:
        parts = parts[:3]
    repl = {
        "PSSA": "pSSA", "AAM": "AAm", "AMPS": "AMPS", "NIPAM": "NIPAM",
        "HMA": "HMA", "TFEMA": "TFEMA", "HA": "HA",
        "TECL": "TECL", "TRCL": "TRCL", "DICL": "DICL",
        "VBA": "VBA", "HEA": "HEA", "DMAA": "DMAA", "MEDSAH": "MEDSAH",
    }
    return "_".join([repl.get(p.upper(), p) for p in parts])

def normalize_variant(v):
    if pd.isna(v):
        return ""
    return str(v).strip().lower().replace("cond_", "").replace("+", "")

def parse_monomer_name(name):
    parts = str(name).split("_")
    return parts[:3] if len(parts) >= 3 else None

def monomer_fraction_vector(name, comp1, comp2, comp3, monomer_list):
    vec = np.zeros(len(monomer_list), dtype=float)
    mons = parse_monomer_name(name)
    if mons is None:
        return vec
    comps = [comp1, comp2, comp3]
    idx_map = {m: i for i, m in enumerate(monomer_list)}
    for m, v in zip(mons, comps):
        if m in idx_map and pd.notna(v):
            vec[idx_map[m]] += float(v)
    return vec

def build_observed_composition_matrix(df_sub, monomer_list):
    rows = []
    for _, r in df_sub.iterrows():
        rows.append(
            monomer_fraction_vector(
                r["Copolymer_Name_norm"],
                r.get("comp_1", np.nan),
                r.get("comp_2", np.nan),
                r.get("comp_3", np.nan),
                monomer_list
            )
        )
    return np.vstack(rows) if len(rows) else np.zeros((0, len(monomer_list)))

def scale01(arr):
    arr = np.asarray(arr, float)
    finite = np.isfinite(arr)
    out = np.full_like(arr, np.nan, dtype=float)
    if finite.sum() == 0:
        return out
    mn, mx = np.nanmin(arr[finite]), np.nanmax(arr[finite])
    if mx == mn:
        out[finite] = 0.0
    else:
        out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def minmax01(x):
    x = np.asarray(x, float)
    finite = np.isfinite(x)
    out = np.zeros_like(x, dtype=float)
    if finite.sum() == 0:
        return out
    mn, mx = np.nanmin(x[finite]), np.nanmax(x[finite])
    if mx <= mn:
        out[finite] = 0.5
    else:
        out[finite] = (x[finite] - mn) / (mx - mn)
    return out

def compute_ilt_dynamics_score_row(r):
    peaks = [
        pd.to_numeric(r.get("Peak1_T2_ms", np.nan), errors="coerce"),
        pd.to_numeric(r.get("Peak2_T2_ms", np.nan), errors="coerce"),
        pd.to_numeric(r.get("Peak3_T2_ms", np.nan), errors="coerce"),
        pd.to_numeric(r.get("Peak4_T2_ms", np.nan), errors="coerce"),
    ]
    peaks = np.array([p for p in peaks if np.isfinite(p) and p > 0], dtype=float)

    logmean = pd.to_numeric(r.get("Weighted_logmean_T2_ms", np.nan), errors="coerce")
    width = pd.to_numeric(r.get("Width_log10T2", np.nan), errors="coerce")
    npeak = pd.to_numeric(r.get("N_detected_peaks", np.nan), errors="coerce")

    mobility = np.log10(logmean) if np.isfinite(logmean) and logmean > 0 else np.nan
    longtail = np.log10(np.max(peaks)) if len(peaks) > 0 else np.nan
    hetero = width if np.isfinite(width) else np.nan
    complex_ = npeak if np.isfinite(npeak) else np.nan

    vals = np.array([mobility, longtail, hetero, complex_], dtype=float)
    if np.all(~np.isfinite(vals)):
        return np.nan

    w = np.array([0.45, 0.20, 0.25, 0.10])
    mask = np.isfinite(vals)
    w_eff = w[mask] / w[mask].sum()
    return float(np.sum(vals[mask] * w_eff))

def sample_conditioned_pool(n, monomer_list, total_active, rng):
    idx = {m: i for i, m in enumerate(monomer_list)}
    X = np.zeros((n, len(monomer_list)), dtype=float)

    hydrophilic_choices = [m for m in HYDROPHILIC if m in idx]
    hydrophobic_choices = [m for m in HYDROPHOBIC if m in idx]
    crosslink_choices = [m for m in CROSSLINKER if m in idx]

    for i in range(n):
        frac = rng.dirichlet([1, 1, 1]) * total_active
        X[i, idx[rng.choice(hydrophilic_choices)]] = frac[0]
        X[i, idx[rng.choice(hydrophobic_choices)]] = frac[1]
        X[i, idx[rng.choice(crosslink_choices)]] = frac[2]

    return X

def X_to_components(x, monomer_list):
    idx = {m: i for i, m in enumerate(monomer_list)}
    hphil = [(m, x[idx[m]]) for m in HYDROPHILIC if m in idx and x[idx[m]] > 0]
    hpho = [(m, x[idx[m]]) for m in HYDROPHOBIC if m in idx and x[idx[m]] > 0]
    cross = [(m, x[idx[m]]) for m in CROSSLINKER if m in idx and x[idx[m]] > 0]

    def pick_best(lst):
        return max(lst, key=lambda t: t[1]) if len(lst) else ("Unknown", 0.0)

    hp, hp_v = pick_best(hphil)
    hb, hb_v = pick_best(hpho)
    cl, cl_v = pick_best(cross)
    return hp, hb, cl, hp_v, hb_v, cl_v

def fit_candidate_feature_models(sub0, X_comp0, numeric_cols):
    models, medians = {}, {}
    for col in numeric_cols:
        y = pd.to_numeric(sub0[col], errors="coerce").values.astype(float)
        mask = np.isfinite(y) & np.isfinite(X_comp0).all(axis=1)
        if mask.sum() >= 6 and np.nanstd(y[mask]) > 1e-12:
            model = GradientBoostingRegressor(random_state=0)
            model.fit(X_comp0[mask], y[mask])
            models[col] = model
            medians[col] = float(np.nanmedian(y[mask]))
        else:
            models[col] = None
            medians[col] = float(np.nanmedian(y[np.isfinite(y)])) if np.isfinite(y).any() else 0.0
    return models, medians

def predict_candidate_numeric_features(X_pool, models, medians, numeric_cols):
    arr = np.zeros((len(X_pool), len(numeric_cols)), dtype=float)
    for j, col in enumerate(numeric_cols):
        model = models.get(col, None)
        if model is not None:
            pred = model.predict(X_pool)
        else:
            pred = np.full(len(X_pool), medians.get(col, 0.0))
        arr[:, j] = np.asarray(pred, dtype=float)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def make_candidate_caption(x, monomer_list, variant_norm, cand_numeric=None, numeric_cols=None):
    hp, hb, cl, hp_v, hb_v, cl_v = X_to_components(x, monomer_list)

    num = {}
    if cand_numeric is not None and numeric_cols is not None:
        num = {c: cand_numeric[i] for i, c in enumerate(numeric_cols)}

    parts = []

    if "s" in variant_norm:
        parts.append(
            f"Structure crosslinked copolymer architecture hydrophilic unit {hp} "
            f"hydrophobic unit {hb} crosslinker {cl} polymer network topology"
        )

    if "n" in variant_norm:
        parts.append("Numeric-only quantitative composition meso-structure interaction and ILT descriptors")

    if "c" in variant_norm:
        parts.append(
            "Chemistry hydration polarity hydrophobicity ionic character aromaticity "
            "fluorination hydrogen bonding network formation"
        )

    if "m" in variant_norm:
        scores = [
            num.get("headgroup_score_norm", np.nan),
            num.get("glycerol_score_norm", np.nan),
            num.get("alkenyl_score_norm", np.nan),
            num.get("alkyl_chain_score_norm", np.nan),
        ]
        motifs = ["headgroup", "glycerol", "alkenyl", "alkyl-chain"]

        if np.isfinite(scores).any():
            dom = motifs[int(np.nanargmax(scores))]
            parts.append(
                f"Meso-structure lipid-associated dominant meso-structural region {dom} "
                "headgroup glycerol alkenyl alkyl-chain association"
            )
        else:
            parts.append("Meso-structure lipid-associated headgroup glycerol alkenyl alkyl-chain association")

    if "i" in variant_norm:
        selectivity = num.get("motif_selectivity_index", np.nan)
        breadth = num.get("motif_breadth_count_0p5", np.nan)

        if np.isfinite(selectivity) and selectivity >= 0.35:
            sel = "highly meso-structure-selective interaction"
        elif np.isfinite(selectivity) and selectivity >= 0.18:
            sel = "moderately meso-structure-selective interaction"
        else:
            sel = "weakly selective distributed interaction"

        if np.isfinite(breadth) and breadth >= 3:
            br = "broad multi-region perturbation"
        elif np.isfinite(breadth) and breadth >= 2:
            br = "dual-region perturbation"
        else:
            br = "localized single-region perturbation"

        parts.append(
            f"Interaction {sel} {br} polymer-lipid localized broad perturbation "
            "meso-structure-dependent interfacial response"
        )

    if "d" in variant_norm:
        wl = num.get("Weighted_logmean_T2_ms", np.nan)
        width = num.get("Width_log10T2", np.nan)
        npeaks = num.get("N_detected_peaks", np.nan)

        dwords = ["Dynamics", "ILT-derived", "mobility", "heterogeneity", "relaxation", "landscape"]

        if np.isfinite(wl):
            if wl >= 100:
                dwords.append("mobile segmental dynamics")
            elif wl >= 10:
                dwords.append("intermediate segmental mobility")
            else:
                dwords.append("constrained segmental dynamics")

        if np.isfinite(width):
            if width >= 1.2:
                dwords.append("heterogeneous broad relaxation distribution")
            elif width >= 0.6:
                dwords.append("moderately heterogeneous relaxation distribution")
            else:
                dwords.append("narrow relaxation distribution")

        if np.isfinite(npeaks):
            if npeaks >= 3:
                dwords.append("multi-component relaxation landscape")
            elif npeaks >= 2:
                dwords.append("two-component relaxation landscape")
            else:
                dwords.append("single dominant relaxation component")

        parts.append(" ".join(dwords))

    if "p" in variant_norm:
        parts.append(
            "Process thermally initiated radical polymerization polar aprotic solvent DMSO "
            "AIBN initiator curing crosslinked polymer network preparation"
        )

    if len(parts) == 0:
        parts.append(f"polymer candidate {hp}_{hb}_{cl}")

    return ". ".join(parts)

def token_from_col(col):
    if col.startswith("TOK_"):
        return col[4:]
    if col.startswith("SP_"):
        return col[3:]
    return col

def normalize_token_text(s):
    s = str(s).lower()
    s = s.replace("▁", " ")
    s = re.sub(r"[^a-z0-9_+\-\.]+", " ", s)
    return s

def captions_to_token_features(captions, feature_cols):
    tokens = [token_from_col(c) for c in feature_cols]
    token_norm = [normalize_token_text(t).strip() for t in tokens]
    X = np.zeros((len(captions), len(feature_cols)), dtype=float)

    for i, cap in enumerate(captions):
        cap_norm = normalize_token_text(cap)
        for j, tok in enumerate(token_norm):
            if tok == "":
                continue
            if " " in tok:
                X[i, j] = cap_norm.count(tok)
            else:
                pattern = r"(?<![a-z0-9_])" + re.escape(tok) + r"(?![a-z0-9_])"
                X[i, j] = len(re.findall(pattern, cap_norm))
    return X

def observed_languange_features(df_sub, feature_cols):
    X = df_sub[feature_cols].copy()
    for c in feature_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    return X.fillna(0.0).values.astype(float)

def observed_numeric_features(df_sub, numeric_cols):
    X = df_sub[numeric_cols].copy()
    for c in numeric_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    med = X.median(axis=0, numeric_only=True).fillna(0.0)
    X = X.fillna(med).fillna(0.0)
    return X.values.astype(float)

def safe_standardize(X_obs, X_cand):
    scaler = StandardScaler()
    X_obs_s = scaler.fit_transform(X_obs)
    X_cand_s = scaler.transform(X_cand)
    return (
        np.nan_to_num(X_obs_s, nan=0.0, posinf=0.0, neginf=0.0),
        np.nan_to_num(X_cand_s, nan=0.0, posinf=0.0, neginf=0.0),
    )

def reduce_joint_pca(X_obs, X_cand, max_dim=32):
    X_all = np.vstack([X_obs, X_cand])
    if X_all.shape[0] <= 2:
        return X_obs, X_cand
    ncomp = min(max_dim, X_all.shape[0] - 1, X_all.shape[1])
    ncomp = max(ncomp, 1)
    Z = PCA(n_components=ncomp, random_state=0).fit_transform(X_all)
    return Z[:len(X_obs)], Z[len(X_obs):]

def normalize_pairwise_distance_scale(X_obs, X_cand):
    if len(X_obs) < 2:
        return X_obs, X_cand
    D = cdist(X_obs, X_obs)
    vals = D[np.triu_indices_from(D, k=1)]
    med = np.nanmedian(vals)
    if not np.isfinite(med) or med <= 1e-12:
        return X_obs, X_cand
    return X_obs / med, X_cand / med

def make_hybrid_repr(X_num_obs_s, X_num_cand_s, X_lang_obs_pca, X_lang_cand_pca):
    X_hybrid_obs = np.hstack([
        NUMERIC_WEIGHT * X_num_obs_s,
        LANGUANGE_WEIGHT * X_lang_obs_pca
    ])
    X_hybrid_cand = np.hstack([
        NUMERIC_WEIGHT * X_num_cand_s,
        LANGUANGE_WEIGHT * X_lang_cand_pca
    ])
    return normalize_pairwise_distance_scale(X_hybrid_obs, X_hybrid_cand)

# ===============================================================
# 4. BO and BLOX
# ===============================================================

def is_non_dominated(Y):
    Y = np.asarray(Y, dtype=float)
    n = Y.shape[0]
    nd = np.ones(n, dtype=bool)

    for i in range(n):
        if not nd[i]:
            continue
        dominates_i = (
            np.all(Y >= Y[i], axis=1)
            & np.any(Y > Y[i], axis=1)
        )
        if np.any(dominates_i):
            nd[i] = False

    return nd

def bo_acquisition(mu_dyn, sd_dyn, mu_meso, sd_meso):
    mu_dyn = np.asarray(mu_dyn, dtype=float)
    mu_meso = np.asarray(mu_meso, dtype=float)
    sd_dyn = np.asarray(sd_dyn, dtype=float)
    sd_meso = np.asarray(sd_meso, dtype=float)

    Y_mu = np.c_[mu_dyn, mu_meso]
    nd_mask = is_non_dominated(Y_mu)

    dyn_s = minmax01(mu_dyn)
    meso_s = minmax01(mu_meso)

    uncertainty_s = minmax01(sd_dyn) + minmax01(sd_meso)
    uncertainty_s = minmax01(uncertainty_s)

    balanced_score = np.sqrt(np.clip(dyn_s, 0, 1) * np.clip(meso_s, 0, 1))
    score = balanced_score + 0.15 * uncertainty_s
    score[~nd_mask] *= 0.25

    return score, nd_mask

def update_available_by_min_distance(available, X_cand_comp, pick, min_dist):
    if min_dist is None or min_dist <= 0:
        available[pick] = False
        return available
    d = np.linalg.norm(X_cand_comp - X_cand_comp[pick], axis=1)
    available = available & (d > min_dist)
    available[pick] = False
    return available

def run_bo(
    X_obs_repr,
    X_cand_repr,
    X_cand_comp,
    y_dyn_obs,
    y_meso_obs,
    n_iter,
    init_idx
):
    Xo = X_obs_repr[init_idx].copy()
    yd = y_dyn_obs[init_idx].copy()
    ym = y_meso_obs[init_idx].copy()

    picked = []
    picked_mu_dyn = []
    picked_mu_meso = []
    picked_is_pareto = []

    available = np.ones(len(X_cand_repr), dtype=bool)

    for _ in range(n_iter):
        cand_idx = np.where(available)[0]
        if len(cand_idx) == 0:
            break

        kernel_d = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=1e-5)
        kernel_m = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=1e-5)

        gp_dyn = GaussianProcessRegressor(kernel=kernel_d, normalize_y=True, random_state=0)
        gp_meso = GaussianProcessRegressor(kernel=kernel_m, normalize_y=True, random_state=1)

        gp_dyn.fit(Xo, yd)
        gp_meso.fit(Xo, ym)

        mu_dyn, sd_dyn = gp_dyn.predict(X_cand_repr[cand_idx], return_std=True)
        mu_meso, sd_meso = gp_meso.predict(X_cand_repr[cand_idx], return_std=True)

        mu_dyn = np.clip(mu_dyn, 0, 1)
        mu_meso = np.clip(mu_meso, 0, 1)

        score, nd_mask = bo_acquisition(mu_dyn, sd_dyn, mu_meso, sd_meso)

        local_pick = int(np.argmax(score))
        pick = cand_idx[local_pick]

        picked.append(X_cand_comp[pick].copy())
        picked_mu_dyn.append(float(mu_dyn[local_pick]))
        picked_mu_meso.append(float(mu_meso[local_pick]))
        picked_is_pareto.append(bool(nd_mask[local_pick]))

        Xo = np.vstack([Xo, X_cand_repr[pick]])
        yd = np.append(yd, mu_dyn[local_pick])
        ym = np.append(ym, mu_meso[local_pick])

        available = update_available_by_min_distance(
            available,
            X_cand_comp,
            pick,
            MIN_COMPOSITION_DISTANCE
        )

    info = pd.DataFrame({
        "pred_mu_dynamics": picked_mu_dyn,
        "pred_mu_mesostructure": picked_mu_meso,
        "selected_from_predicted_front": picked_is_pareto,
    })

    return np.array(picked, dtype=float), info

def run_blox(X_obs_repr, X_cand_repr, X_cand_comp, n_iter, init_idx, rng):
    observed_repr = X_obs_repr[init_idx].copy()
    picked = []
    available = np.ones(len(X_cand_repr), dtype=bool)

    for _ in range(n_iter):
        cand_available = np.where(available)[0]
        if len(cand_available) == 0:
            break

        if len(cand_available) > N_CAND:
            cand_idx = rng.choice(cand_available, size=N_CAND, replace=False)
        else:
            cand_idx = cand_available

        D = cdist(X_cand_repr[cand_idx], observed_repr)
        novelty = np.sum(np.exp(-D), axis=1)
        pick = cand_idx[int(np.argmin(novelty))]

        picked.append(X_cand_comp[pick].copy())
        observed_repr = np.vstack([observed_repr, X_cand_repr[pick]])

        available = update_available_by_min_distance(
            available,
            X_cand_comp,
            pick,
            MIN_COMPOSITION_DISTANCE
        )

    return np.array(picked, dtype=float)

# ===============================================================
# 5. Violin, JSD, and Word export helpers
# ===============================================================

def composition_diversity_metrics(X_prop, monomer_order=None, eps=1e-12):
    if X_prop is None or len(X_prop) == 0:
        return {"JSD_diversity": np.nan}

    X = np.asarray(X_prop, dtype=float)

    if monomer_order is not None:
        idx_map = {m: i for i, m in enumerate(ALL_MONOMERS)}
        cols = [idx_map[m] for m in monomer_order if m in idx_map]
        X = X[:, cols]

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0

    row_sum = X.sum(axis=1, keepdims=True)
    X_valid = X[row_sum[:, 0] > eps]

    if len(X_valid) < 2:
        return {"JSD_diversity": 0.0}

    P = X_valid / (X_valid.sum(axis=1, keepdims=True) + eps)

    def js_distance(p, q):
        m = 0.5 * (p + q)

        def kl(a, b):
            mask = a > eps
            return np.sum(a[mask] * np.log((a[mask] + eps) / (b[mask] + eps)))

        jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
        return np.sqrt(jsd) / np.sqrt(np.log(2))

    vals = []
    for i in range(len(P)):
        for j in range(i + 1, len(P)):
            vals.append(js_distance(P[i], P[j]))

    return {"JSD_diversity": float(np.mean(vals))}

def violin_panel_from_proposals_with_jsd(ax, X_prop, title, monomer_order, color):
    metrics = composition_diversity_metrics(X_prop, monomer_order=monomer_order)

    idx_map = {m: i for i, m in enumerate(ALL_MONOMERS)}
    data, labels = [], []

    for mon in monomer_order:
        if mon not in idx_map:
            continue
        vals = np.asarray(X_prop[:, idx_map[mon]], dtype=float)
        vals = vals[np.isfinite(vals)]
        vals = vals[vals > 0]
        if len(vals) > 0:
            data.append(vals)
            labels.append(mon)

    if len(data) == 0:
        ax.text(0.5, 0.5, "No positive fractions", ha="center", va="center")
        return metrics

    parts = ax.violinplot(data, vert=False, showextrema=False, showmedians=False)

    for body in parts["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_alpha(0.62)

    for i, vals in enumerate(data, start=1):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        ax.plot([q1, q3], [i, i], lw=4, color="black", solid_capstyle="round")
        ax.scatter(med, i, s=24, color="black", zorder=3)

    ax.text(
        0.98, 0.98,
        f"JSD div. = {metrics['JSD_diversity']:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="gray", alpha=0.85)
    )

    ax.set_title(title, fontsize=SUBTITLE_FS)
    ax.set_xlim(0, 1.0)
    ax.grid(axis="x", alpha=0.28)
    ax.set_yticks(np.arange(1, len(labels) + 1))
    ax.set_yticklabels(labels, fontsize=TICK_FS)
    ax.set_xlabel("Composition fraction", fontsize=LABEL_FS)
    ax.tick_params(axis="both", labelsize=TICK_FS)

    return metrics

def plot_violin_only_for_variant(variant_norm, selected, repeat=0):
    variant_label = VARIANT_LABEL_MAP.get(variant_norm, variant_norm.upper())

    fig, axes = plt.subplots(2, 2, figsize=(7.6, 9.8), sharex=True)
    axes = axes.ravel()
    div_rows = []

    for ax, method in zip(axes, METHODS):
        metrics = violin_panel_from_proposals_with_jsd(
            ax=ax,
            X_prop=selected[method],
            title=method,
            monomer_order=VIOLIN_MONOMERS,
            color=METHOD_COLORS[method],
        )
        div_rows.append({
            "variant": variant_norm,
            "variant_label": variant_label,
            "repeat": repeat,
            "method": method,
            **metrics
        })

    axes[0].set_ylabel("Monomer", fontsize=LABEL_FS)
    axes[2].set_ylabel("Monomer", fontsize=LABEL_FS)

    fig.suptitle(
        f"{variant_label}: composition diversity under Dynamics × Meso-structure exploration",
        fontsize=TITLE_FS,
        y=0.995
    )

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    png = os.path.join(OUTDIR, f"BO_consistent_Violin_JSD_{variant_norm}_repeat{repeat}.png")
    pdf = os.path.join(OUTDIR, f"BO_consistent_Violin_JSD_{variant_norm}_repeat{repeat}.pdf")

    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    GENERATED_FIGURES.append(png)

    plt.show()
    plt.close()

    return div_rows

def plot_jsd_diversity_bar(diversity_df, variant_norm):
    variant_label = VARIANT_LABEL_MAP.get(variant_norm, variant_norm.upper())
    sub = diversity_df[diversity_df["variant"] == variant_norm].copy()

    agg = (
        sub.groupby("method")
        .agg(mean=("JSD_diversity", "mean"), std=("JSD_diversity", "std"))
        .reindex(METHODS)
    )

    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    x = np.arange(len(METHODS))
    means = agg["mean"].values
    stds = agg["std"].fillna(0).values

    ax.bar(
        x,
        means,
        yerr=stds,
        capsize=5,
        color=[METHOD_COLORS[m] for m in METHODS],
        edgecolor="black",
        alpha=0.88
    )

    for i, m in enumerate(means):
        if np.isfinite(m):
            ax.text(i, m + 0.025, f"{m:.2f}", ha="center", va="bottom", fontsize=11)

    ax.set_xticks(x)
    ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=11)
    ax.set_ylabel("JSD diversity", fontsize=14)
    ax.set_title(f"{variant_label}: composition diversity under BO-consistent exploration", fontsize=15)
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()

    png = os.path.join(OUTDIR, f"BO_consistent_JSD_bar_{variant_norm}.png")
    pdf = os.path.join(OUTDIR, f"BO_consistent_JSD_bar_{variant_norm}.pdf")

    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    GENERATED_FIGURES.append(png)

    plt.show()
    plt.close()

def save_proposals_csv(X_sel, method, variant_norm, repeat):
    prop_df = pd.DataFrame(X_sel, columns=ALL_MONOMERS)
    prop_df.insert(0, "proposal_id", np.arange(1, len(prop_df) + 1))
    prop_df.insert(0, "repeat", repeat)
    prop_df.insert(0, "method", method)
    prop_df.insert(0, "variant", variant_norm)
    prop_df.insert(1, "variant_label", VARIANT_LABEL_MAP.get(variant_norm, variant_norm.upper()))

    safe_method = method.replace("+", "plus").replace("-", "_")
    out = os.path.join(OUTDIR, f"Proposals_{safe_method}_{variant_norm}_repeat{repeat}.csv")
    prop_df.to_csv(out, index=False, encoding="utf-8-sig")
    return prop_df

def export_figures_to_a4_word(fig_paths, word_path):
    doc = Document()

    section = doc.sections[0]
    section.page_width = Inches(8.27)
    section.page_height = Inches(11.69)
    section.top_margin = Inches(0.45)
    section.bottom_margin = Inches(0.45)
    section.left_margin = Inches(0.45)
    section.right_margin = Inches(0.45)

    usable_width = 8.27 - 0.45 - 0.45

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title.add_run("BO-consistent composition diversity figures")
    run.bold = True
    run.font.size = Pt(14)

    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run("A4 layout for Word insertion")
    r.font.size = Pt(9)

    for i, fig_path in enumerate(fig_paths):
        if not os.path.exists(fig_path):
            continue

        if i > 0:
            doc.add_section(WD_SECTION.NEW_PAGE)
            section = doc.sections[-1]
            section.page_width = Inches(8.27)
            section.page_height = Inches(11.69)
            section.top_margin = Inches(0.45)
            section.bottom_margin = Inches(0.45)
            section.left_margin = Inches(0.45)
            section.right_margin = Inches(0.45)

        cap = doc.add_paragraph()
        cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
        rr = cap.add_run(os.path.basename(fig_path))
        rr.font.size = Pt(8)

        para = doc.add_paragraph()
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        para.add_run().add_picture(fig_path, width=Inches(usable_width))

    doc.save(word_path)
    print("A4 Word file saved:", word_path)

# ===============================================================
# 6. Preprocess
# ===============================================================

df["Copolymer_Name_norm"] = df["Copolymer_Name"].apply(normalize_name)
df["variant_norm"] = df["variant"].apply(normalize_variant)
df = df[df["variant_norm"].isin(TARGET_VARIANTS)].copy().reset_index(drop=True)

if len(df) == 0:
    raise ValueError("No rows found for TARGET_VARIANTS.")

df["Dynamics_raw_local"] = df.apply(compute_ilt_dynamics_score_row, axis=1)

source_df = df[df["variant_norm"] == DYNAMICS_SOURCE_VARIANT].copy()
if len(source_df) == 0 or source_df["Dynamics_raw_local"].notna().sum() == 0:
    print(f"[WARN] No valid dynamics for {DYNAMICS_SOURCE_VARIANT}; using all variants.")
    source_df = df.copy()

source_dyn = (
    source_df[["Copolymer_Name", "Dynamics_raw_local"]]
    .dropna(subset=["Dynamics_raw_local"])
    .groupby("Copolymer_Name", as_index=False)["Dynamics_raw_local"]
    .mean()
    .rename(columns={"Dynamics_raw_local": "Dynamics_raw_shared"})
)

df = df.merge(source_dyn, on="Copolymer_Name", how="left")

finite = np.isfinite(df["Dynamics_raw_shared"].values.astype(float))
df["Dynamics"] = np.nan
df.loc[finite, "Dynamics"] = scale01(df.loc[finite, "Dynamics_raw_shared"].values)

variants = [v for v in TARGET_VARIANTS if v in sorted(df["variant_norm"].unique())]

print("Variants used:", variants)
print("Variant labels:", [VARIANT_LABEL_MAP.get(v, v.upper()) for v in variants])
print("Rows used:", df.shape)

# ===============================================================
# 7. Main loop
# ===============================================================

DIVERSITY_ROWS = []
PROP_ALL_ROWS = []
BO_INFO_ROWS = []

for variant_norm in variants:

    variant_label = VARIANT_LABEL_MAP.get(variant_norm, variant_norm.upper())
    print(f"\n=== Variant: {variant_label} ===")

    sub0 = df[df["variant_norm"] == variant_norm].copy().reset_index(drop=True)

    X_comp0 = build_observed_composition_matrix(sub0, ALL_MONOMERS)
    valid_comp = X_comp0.sum(axis=1) > 0

    sub0 = sub0.loc[valid_comp].reset_index(drop=True)
    X_comp0 = X_comp0[valid_comp]

    dyn0 = sub0["Dynamics"].values.astype(float)

    meso0_raw = pd.to_numeric(sub0["motif_selectivity_index"], errors="coerce").values.astype(float)

    if np.isfinite(meso0_raw).any():
        meso_fill = np.nanmedian(meso0_raw[np.isfinite(meso0_raw)])
        meso0_filled = np.where(np.isfinite(meso0_raw), meso0_raw, meso_fill)
        meso0 = scale01(meso0_filled)
    else:
        meso0 = np.zeros_like(dyn0)

    valid = (
        np.isfinite(dyn0)
        & np.isfinite(meso0)
        & np.isfinite(X_comp0).all(axis=1)
    )

    sub0 = sub0.loc[valid].reset_index(drop=True)
    X_comp0 = X_comp0[valid]
    dyn0 = dyn0[valid]
    meso0 = meso0[valid]

    if len(sub0) < 8:
        print("Skipped due to too few valid samples.")
        continue

    X_lang_obs_raw0 = observed_languange_features(sub0, LANGUANGE_FEATURE_COLS)
    X_num_obs_raw0 = observed_numeric_features(sub0, NUMERIC_REPR_COLS_BASE)

    total_active_obs = np.nanmedian(
        sub0[["comp_1", "comp_2", "comp_3"]].sum(axis=1).values.astype(float)
    )

    if not np.isfinite(total_active_obs) or total_active_obs <= 0:
        total_active_obs = TOTAL_ACTIVE_DEFAULT

    numeric_models, numeric_medians = fit_candidate_feature_models(
        sub0,
        X_comp0,
        NUMERIC_REPR_COLS_BASE
    )

    for repeat in range(N_REPEATS):

        rng = np.random.default_rng(1000 + repeat)

        POOL = sample_conditioned_pool(
            N_POOL,
            ALL_MONOMERS,
            total_active_obs,
            rng
        )

        X_num_cand_raw = predict_candidate_numeric_features(
            POOL,
            numeric_models,
            numeric_medians,
            NUMERIC_REPR_COLS_BASE
        )

        cand_captions = [
            make_candidate_caption(
                x,
                ALL_MONOMERS,
                variant_norm,
                cand_numeric=X_num_cand_raw[i],
                numeric_cols=NUMERIC_REPR_COLS_BASE
            )
            for i, x in enumerate(POOL)
        ]

        X_lang_cand_raw = captions_to_token_features(cand_captions, LANGUANGE_FEATURE_COLS)

        X_num_obs_s, X_num_cand_s = safe_standardize(X_num_obs_raw0, X_num_cand_raw)
        X_num_obs, X_num_cand = normalize_pairwise_distance_scale(X_num_obs_s, X_num_cand_s)

        X_lang_obs_s, X_lang_cand_s = safe_standardize(X_lang_obs_raw0, X_lang_cand_raw)
        X_lang_obs_pca, X_lang_cand_pca = reduce_joint_pca(
            X_lang_obs_s,
            X_lang_cand_s,
            max_dim=32
        )

        X_hybrid_obs, X_hybrid_cand = make_hybrid_repr(
            X_num_obs_s,
            X_num_cand_s,
            X_lang_obs_pca,
            X_lang_cand_pca
        )

        init_n = min(5, len(dyn0))
        init_idx = rng.choice(len(dyn0), size=init_n, replace=False)

        selected = {}

        selected["BO-Numerical"], info_num = run_bo(
            X_obs_repr=X_num_obs,
            X_cand_repr=X_num_cand,
            X_cand_comp=POOL,
            y_dyn_obs=dyn0,
            y_meso_obs=meso0,
            n_iter=BO_ITERS,
            init_idx=init_idx
        )

        info_num.insert(0, "repeat", repeat)
        info_num.insert(0, "method", "BO-Numerical")
        info_num.insert(0, "variant", variant_norm)
        BO_INFO_ROWS.append(info_num)

        selected["BO-N+Languange"], info_lang = run_bo(
            X_obs_repr=X_hybrid_obs,
            X_cand_repr=X_hybrid_cand,
            X_cand_comp=POOL,
            y_dyn_obs=dyn0,
            y_meso_obs=meso0,
            n_iter=BO_ITERS,
            init_idx=init_idx
        )

        info_lang.insert(0, "repeat", repeat)
        info_lang.insert(0, "method", "BO-N+Languange")
        info_lang.insert(0, "variant", variant_norm)
        BO_INFO_ROWS.append(info_lang)

        selected["BLOX-Numerical"] = run_blox(
            X_obs_repr=X_num_obs,
            X_cand_repr=X_num_cand,
            X_cand_comp=POOL,
            n_iter=BLOX_ITERS,
            init_idx=init_idx,
            rng=rng
        )

        selected["BLOX-N+Languange"] = run_blox(
            X_obs_repr=X_hybrid_obs,
            X_cand_repr=X_hybrid_cand,
            X_cand_comp=POOL,
            n_iter=BLOX_ITERS,
            init_idx=init_idx,
            rng=rng
        )

        for method, X_sel in selected.items():
            prop_df = save_proposals_csv(
                X_sel=X_sel,
                method=method,
                variant_norm=variant_norm,
                repeat=repeat
            )
            PROP_ALL_ROWS.append(prop_df)

            metrics = composition_diversity_metrics(
                X_sel,
                monomer_order=VIOLIN_MONOMERS
            )

            DIVERSITY_ROWS.append({
                "variant": variant_norm,
                "variant_label": variant_label,
                "repeat": repeat,
                "method": method,
                **metrics
            })

        if repeat == 0:
            plot_violin_only_for_variant(
                variant_norm=variant_norm,
                selected=selected,
                repeat=repeat
            )

# ===============================================================
# 8. Save CSVs and plots
# ===============================================================

diversity_df = pd.DataFrame(DIVERSITY_ROWS)

diversity_df.to_csv(
    os.path.join(OUTDIR, "BO_consistent_monomer_JSD_diversity_all_repeats.csv"),
    index=False,
    encoding="utf-8-sig"
)

agg_df = (
    diversity_df
    .groupby(["variant", "variant_label", "method"], as_index=False)
    .agg(
        JSD_diversity_mean=("JSD_diversity", "mean"),
        JSD_diversity_std=("JSD_diversity", "std"),
    )
)

agg_df.to_csv(
    os.path.join(OUTDIR, "BO_consistent_monomer_JSD_diversity_aggregated.csv"),
    index=False,
    encoding="utf-8-sig"
)

if len(PROP_ALL_ROWS) > 0:
    all_prop_df = pd.concat(PROP_ALL_ROWS, axis=0, ignore_index=True)
    all_prop_df.to_csv(
        os.path.join(OUTDIR, "BO_consistent_all_proposals_composition_vectors.csv"),
        index=False,
        encoding="utf-8-sig"
    )

if len(BO_INFO_ROWS) > 0:
    bo_info_df = pd.concat(BO_INFO_ROWS, axis=0, ignore_index=True)
    bo_info_df.to_csv(
        os.path.join(OUTDIR, "BO_selection_info_Dynamics_Mesostructure.csv"),
        index=False,
        encoding="utf-8-sig"
    )

for variant_norm in variants:
    if variant_norm in diversity_df["variant"].unique():
        plot_jsd_diversity_bar(diversity_df, variant_norm)

# ===============================================================
# 9. A4 Word export
# ===============================================================

unique_figures = []
for p in GENERATED_FIGURES:
    if p not in unique_figures and os.path.exists(p):
        unique_figures.append(p)

export_figures_to_a4_word(unique_figures, WORD_PATH)

# ===============================================================
# 10. ZIP export
# ===============================================================

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTDIR):
        for file in files:
            fp = os.path.join(root, file)
            arcname = os.path.relpath(fp, OUTDIR)
            zf.write(fp, arcname=arcname)

    if os.path.exists(WORD_PATH):
        zf.write(WORD_PATH, arcname=os.path.basename(WORD_PATH))

print("\nDone.")
print("Output folder:", OUTDIR)
print("A4 Word file :", WORD_PATH)
print("ZIP          :", ZIP_PATH)

try:
    from google.colab import files
    files.download(WORD_PATH)
    files.download(ZIP_PATH)
except Exception:
    pass